In [1]:
import h5py
import torch
from torch import nn
from torchsummary import summary
from torch.utils.data import DataLoader
from torch.nn.utils import clip_grad_norm_
from chess2.bot import ChessDataset
from chess2.bot import NeuralNetwork

In [2]:
# Hyperparameters
BATCH_SIZE = 64
EPOCHS = 25 #150
LEARNING_RATE = 1e-3 #1e-4 was good
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 5

In [3]:
train_data = ChessDataset("/Users/jonas/coding/python/chess2/src/chess2/bot/data_leela/chess_data_list.pkl", 0, 800000)
validation_data = ChessDataset("/Users/jonas/coding/python/chess2/src/chess2/bot/data_leela/chess_data_list.pkl", 800000, 1000000)

In [4]:
train_dataloader = DataLoader(train_data, BATCH_SIZE, shuffle=True)
validation_dataloader = DataLoader(validation_data, BATCH_SIZE, shuffle=True)

In [5]:
device = torch.device("cpu")
model = NeuralNetwork().to(device)

In [6]:
loss_policy = nn.CrossEntropyLoss()

In [7]:
optimizer = torch.optim.SGD(
    params=model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    momentum=0.9,
    nesterov=True
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=3
)

# scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
#     optimizer=optimizer,
#     T_0=10,         # Restart after 10 epochs
#     T_mult=2,       # Restart time every time
#     eta_min=1e-6    # minimum learning rate
# )

In [9]:
def train_loop(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)

    # Set the model to training mode - important for batch normalization and dropout layers
    model.train()

    for batch, (boards, flags, prob_idx) in enumerate(dataloader):
        if any(i > 1858 for i in prob_idx):
            raise ValueError("Invalid Policy Index")
        move_pred = model(boards, flags)
        # move_pred = move_pred.masked_fill(~moves_mask.bool(), -1e9)

        optimizer.zero_grad()

        loss = loss_fn(move_pred, prob_idx)

        loss.backward()
        clip_grad_norm_(model.parameters(), max_norm=1)
        optimizer.step()

        if batch % 100 == 0:
            loss, current = loss.item(), batch * BATCH_SIZE
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")

In [10]:
def validation_loop(dataloader, model, loss_fn):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    validation_loss, correct = 0, 0

    # Set the model in evaluation mode
    model.eval()

    # no_grad to suppress gradient computation
    with torch.no_grad():
        for (boards, flags, prob_idx) in dataloader:
            move_pred = model(boards, flags)

            validation_loss += loss_fn(move_pred, prob_idx).item()
            correct += (move_pred.argmax(dim=1).long() == prob_idx).type(torch.float).sum().item()

    validation_loss /= num_batches
    correct /= size

    print(f"Validation Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss per batch: {validation_loss:>8f} \n")

    return validation_loss

In [17]:
model.load_state_dict(torch.load("/Users/jonas/coding/python/chess2/src/chess2/bot/saved_models/model_new.pth", weights_only=True))

<All keys matched successfully>

In [18]:
for t in range(EPOCHS):
    print(f"Epoch {t+1}\n-------------------------------")
    train_loop(train_dataloader, model, loss_policy, optimizer) # train_dataloader instead of test_dataloader
    val_loss = validation_loop(validation_dataloader, model, loss_policy)
    scheduler.step(val_loss)
    print("LR:", optimizer.param_groups[0]['lr'])
print("Done!")

Epoch 1
-------------------------------
loss: 4.201434  [    0/800000]
loss: 4.198544  [ 6400/800000]
loss: 4.060821  [12800/800000]
loss: 4.073123  [19200/800000]
loss: 3.550499  [25600/800000]
loss: 3.818843  [32000/800000]
loss: 4.255689  [38400/800000]
loss: 3.943812  [44800/800000]
loss: 4.053123  [51200/800000]
loss: 3.695824  [57600/800000]
loss: 3.945966  [64000/800000]
loss: 3.758389  [70400/800000]
loss: 3.660995  [76800/800000]
loss: 4.097519  [83200/800000]
loss: 3.958268  [89600/800000]
loss: 3.542430  [96000/800000]
loss: 3.507915  [102400/800000]
loss: 3.927919  [108800/800000]
loss: 3.800682  [115200/800000]
loss: 3.957892  [121600/800000]
loss: 4.293784  [128000/800000]
loss: 3.716395  [134400/800000]
loss: 3.611353  [140800/800000]
loss: 4.171325  [147200/800000]
loss: 3.245186  [153600/800000]
loss: 4.130786  [160000/800000]
loss: 3.587457  [166400/800000]
loss: 3.862064  [172800/800000]
loss: 3.894731  [179200/800000]
loss: 3.895491  [185600/800000]
loss: 3.893470  

KeyboardInterrupt: 

In [19]:
torch.save(model.state_dict(), '/Users/jonas/coding/python/chess2/src/chess2/bot/saved_models/model_new.pth')